# Advanced Backtesting: Iterative Backtesting ("event-driven") 

## A first Intuition on Iterative Backtesting (Part 1)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")

In [ ]:
data = pd.read_csv("detailed.csv", parse_dates = ["time"], index_col = "time")
data

In [ ]:
data = data.round(5)

In [ ]:
data.price.plot(figsize = (12, 8))
plt.show()

In [ ]:
data.spread.hist(bins = 100, figsize = (12, 8))
plt.show()

In [ ]:
for i in range(10):
    print(i)

In [ ]:
for bar in range(10): # interate over the first 10 bars/candles 
    print(bar, data.index[bar].date(), data.price.iloc[bar], data.spread.iloc[bar], sep = " | ")

In [ ]:
import time

In [ ]:
for bar in range(10):
    print(bar, data.index[bar].date(), data.price.iloc[bar], data.spread.iloc[bar], sep = " | ")
    time.sleep(1)

## A first Intuition on Iterative Backtesting (Part 2)

In [ ]:
data

In [ ]:
sma_s = 50
sma_l = 200

In [ ]:
data["SMA_S"] = data.price.rolling(sma_s).mean()
data["SMA_L"] = data.price.rolling(sma_l).mean()

In [ ]:
data.dropna(inplace = True)

In [ ]:
data

In [ ]:
position = 0 # we start with neutral position

In [ ]:
len(data)

In [ ]:
for bar in range(len(data)):
    if data["SMA_S"].iloc[bar] > data["SMA_L"].iloc[bar]:
        if position in [0, -1]:
            print("{}: Go Long  | Price: {} | Spread: {}".format(data.index[bar].date(), data.price.iloc[bar], data.spread.iloc[bar]))
            position = 1
    elif data["SMA_S"].iloc[bar] < data["SMA_L"].iloc[bar]:
        if position in [0, 1]:
            print("{}: Go Short | Price: {} | Spread: {}".format(data.index[bar].date(), data.price.iloc[bar], data.spread.iloc[bar]))
            position = -1

## Creating an Iterative Base Class (Part 1)

In [ ]:
class IterativeBase():

    def __init__(self, symbol, start, end, amount):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.initial_balance = amount
        self.current_balance = amount
        self.get_data()

    def get_data(self):
        raw = pd.read_csv("detailed.csv", parse_dates = ["time"], index_col = "time").dropna()
        raw = raw.loc[self.start:self.end].copy()
        raw["returns"] = np.log(raw.price / raw.price.shift(1))
        self.data = raw

    def plot_data(self, cols = None):  
        if cols is None:
            cols = "price"
        self.data[cols].plot(figsize = (12, 8), title = self.symbol)

In [ ]:
bc = IterativeBase("EURUSD", "2006-12-31", "2020-06-30", 100000)

In [ ]:
bc.data

In [ ]:
bc.plot_data()

In [ ]:
bc.plot_data(cols = "spread")

## Creating an Iterative Base Class (Part 2)

In [ ]:
class IterativeBase():

    def __init__(self, symbol, start, end, amount):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.initial_balance = amount
        self.current_balance = amount
        self.get_data()

    def get_data(self):
        raw = pd.read_csv("detailed.csv", parse_dates = ["time"], index_col = "time").dropna()
        raw = raw.loc[self.start:self.end]
        raw["returns"] = np.log(raw.price / raw.price.shift(1))
        self.data = raw

    def plot_data(self, cols = None):  
        if cols is None:
            cols = "price"
        self.data[cols].plot(figsize = (12, 8), title = self.symbol)
    
    def get_values(self, bar):
        date = str(self.data.index[bar].date())
        price = round(self.data.price.iloc[bar], 5)
        spread = round(self.data.spread.iloc[bar], 5)
        #returns = round(self.data.returns.iloc[bar], 5)
        return date, price, spread#, returns

In [ ]:
bc = IterativeBase("EURUSD", "2006-12-31", "2020-06-30", 100000)

In [ ]:
bc.data

In [ ]:
bc.get_values(0)

In [ ]:
bc.get_values(100)

In [ ]:
bc.get_values(-1)

## Creating an Iterative Base Class (Part 3)

In [ ]:
class IterativeBase():

    def __init__(self, symbol, start, end, amount):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.initial_balance = amount
        self.current_balance = amount
        self.get_data()

    def get_data(self):
        raw = pd.read_csv("detailed.csv", parse_dates = ["time"], index_col = "time").dropna()
        raw = raw.loc[self.start:self.end]
        raw["returns"] = np.log(raw.price / raw.price.shift(1))
        self.data = raw

    def plot_data(self, cols = None):  
        if cols is None:
            cols = "price"
        self.data[cols].plot(figsize = (12, 8), title = self.symbol)
    
    def get_values(self, bar):
        date = str(self.data.index[bar].date())
        price = round(self.data.price.iloc[bar], 5)
        spread = round(self.data.spread.iloc[bar], 5)
        return date, price, spread
    
    def print_current_balance(self, bar):
        date, price, spread = self.get_values(bar)
        print("{} | Current Balance: {}".format(date, round(self.current_balance, 2)))

In [ ]:
bc = IterativeBase("EURUSD", "2006-12-31", "2020-06-30", 100000)

In [ ]:
bc.data

In [ ]:
bc.print_current_balance(0)

In [ ]:
bc.print_current_balance(100)

In [ ]:
bc.print_current_balance(-1)

## Creating an Iterative Base Class (Part 4)

In [ ]:
class IterativeBase():

    def __init__(self, symbol, start, end, amount):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.initial_balance = amount
        self.current_balance = amount
        self.units = 0
        self.trades = 0
        self.get_data()

    def get_data(self):
        raw = pd.read_csv("detailed.csv", parse_dates = ["time"], index_col = "time").dropna()
        raw = raw.loc[self.start:self.end]
        raw["returns"] = np.log(raw.price / raw.price.shift(1))
        self.data = raw

    def plot_data(self, cols = None):  
        if cols is None:
            cols = "price"
        self.data[cols].plot(figsize = (12, 8), title = self.symbol)
    
    def get_values(self, bar):
        date = str(self.data.index[bar].date())
        price = round(self.data.price.iloc[bar], 5)
        spread = round(self.data.spread.iloc[bar], 5)
        return date, price, spread
    
    def print_current_balance(self, bar):
        date, price, spread = self.get_values(bar)
        print("{} | Current Balance: {}".format(date, round(self.current_balance, 2)))
        
    def buy_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance -= units * price # reduce cash balance by "purchase price"
        self.units += units
        self.trades += 1
        print("{} |  Buying {} for {}".format(date, units, round(price, 5)))

In [ ]:
bc = IterativeBase("EURUSD", "2006-12-31", "2020-06-30", 100000)

In [ ]:
bc.print_current_balance(0)

In [ ]:
bc.data

In [ ]:
bc.buy_instrument(0, units = 1000)

In [ ]:
bc.units

In [ ]:
bc.print_current_balance(0)

In [ ]:
100000 - 1000 * 1.31985

In [ ]:
bc.buy_instrument(1, amount = 5000)

In [ ]:
int(5000 / 1.32734)

In [ ]:
bc.print_current_balance(1)

## Creating an Iterative Base Class (Part 5)

In [4]:
class IterativeBase():

    def __init__(self, symbol, start, end, amount):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.initial_balance = amount
        self.current_balance = amount
        self.units = 0
        self.trades = 0 
        self.get_data()

    def get_data(self):
        raw = pd.read_csv("detailed.csv", parse_dates = ["time"], index_col = "time").dropna()
        raw = raw.loc[self.start:self.end]
        raw["returns"] = np.log(raw.price / raw.price.shift(1))
        self.data = raw

    def plot_data(self, cols = None):  
        if cols is None:
            cols = "price"
        self.data[cols].plot(figsize = (12, 8), title = self.symbol)
    
    def get_values(self, bar):
        date = str(self.data.index[bar].date())
        price = round(self.data.price.iloc[bar], 5)
        spread = round(self.data.spread.iloc[bar], 5)
        return date, price, spread
    
    def print_current_balance(self, bar):
        date, price, spread = self.get_values(bar)
        print("{} | Current Balance: {}".format(date, round(self.current_balance, 2)))
        
    def buy_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance -= units * price # reduce cash balance by "purchase price"
        self.units += units
        self.trades += 1
        print("{} |  Buying {} for {}".format(date, units, round(price, 5)))
    
    def print_current_position_value(self, bar):
        date, price, spread = self.get_values(bar)
        cpv = self.units * price
        print("{} |  Current Position Value = {}".format(date, round(cpv, 2)))
    
    def print_current_nav(self, bar):
        date, price, spread = self.get_values(bar)
        nav = self.current_balance + self.units * price
        print("{} |  Net Asset Value = {}".format(date, round(nav, 2)))

In [5]:
bc = IterativeBase("EURUSD", "2006-12-31", "2020-06-30", 100000)

In [6]:
bc.print_current_balance(0)

2006-12-31 | Current Balance: 100000


In [7]:
bc.buy_instrument(0, units = 1000)

2006-12-31 |  Buying 1000 for 1.31985


In [8]:
bc.units

1000

In [9]:
bc.print_current_balance(0)

2006-12-31 | Current Balance: 98680.15


In [10]:
bc.print_current_position_value(0)

2006-12-31 |  Current Position Value = 1319.85


In [12]:
bc.print_current_nav(0)

2006-12-31 |  Net Asset Value = 100000.0


In [13]:
bc.print_current_position_value(1)

2007-01-01 |  Current Position Value = 1327.34


In [14]:
bc.print_current_nav(1)

2007-01-01 |  Net Asset Value = 100007.49


In [15]:
bc.buy_instrument(1, amount = 5000)

2007-01-01 |  Buying 3766 for 1.32734


In [16]:
bc.print_current_balance(1)

2007-01-01 | Current Balance: 93681.39


In [17]:
bc.print_current_position_value(1)

2007-01-01 |  Current Position Value = 6326.1


In [22]:
bc.print_current_nav(1)

2007-01-01 |  Net Asset Value = 100007.49


In [31]:
bc.print_current_balance(2)

2007-04-22 | Current Balance: 93681.39


In [24]:
bc.print_current_position_value(2)

2007-01-02 |  Current Position Value = 6276.25


In [32]:
bc.print_current_nav(2)

2007-01-02 |  Net Asset Value = 99957.64


In [33]:
bc.print_current_nav(-1)

2020-06-29 |  Net Asset Value = 99035.08


In [34]:
bc.print_current_position_value(-1)

2020-06-29 |  Current Position Value = 5353.7


In [ ]:
bc.data

## Creating an Iterative Base Class (Part 6)

In [35]:
class IterativeBase():

    def __init__(self, symbol, start, end, amount):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.initial_balance = amount
        self.current_balance = amount
        self.units = 0
        self.trades = 0 
        self.get_data()

    def get_data(self):
        raw = pd.read_csv("detailed.csv", parse_dates = ["time"], index_col = "time").dropna()
        raw = raw.loc[self.start:self.end]
        raw["returns"] = np.log(raw.price / raw.price.shift(1))
        self.data = raw

    def plot_data(self, cols = None):  
        if cols is None:
            cols = "price"
        self.data[cols].plot(figsize = (12, 8), title = self.symbol)
    
    def get_values(self, bar):
        date = str(self.data.index[bar].date())
        price = round(self.data.price.iloc[bar], 5)
        spread = round(self.data.spread.iloc[bar], 5)
        return date, price, spread
    
    def print_current_balance(self, bar):
        date, price, spread = self.get_values(bar)
        print("{} | Current Balance: {}".format(date, round(self.current_balance, 2)))
        
    def buy_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance -= units * price # reduce cash balance by "purchase price"
        self.units += units
        self.trades += 1
        print("{} |  Buying {} for {}".format(date, units, round(price, 5)))
    
    def sell_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance += units * price # increases cash balance by "purchase price"
        self.units -= units
        self.trades += 1
        print("{} |  Selling {} for {}".format(date, units, round(price, 5)))
    
    def print_current_position_value(self, bar):
        date, price, spread = self.get_values(bar)
        cpv = self.units * price
        print("{} |  Current Position Value = {}".format(date, round(cpv, 2)))
    
    def print_current_nav(self, bar):
        date, price, spread = self.get_values(bar)
        nav = self.current_balance + self.units * price
        print("{} |  Net Asset Value = {}".format(date, round(nav, 2)))

In [36]:
bc = IterativeBase("EURUSD", "2006-12-31", "2020-06-30", 100000)

In [37]:
bc.buy_instrument(0, units = 10000) # go long 10,000 units

2006-12-31 |  Buying 10000 for 1.31985


In [38]:
bc.units

10000

In [ ]:
bc.print_current_position_value(1)

In [ ]:
bc.print_current_nav(1)

In [ ]:
bc.sell_instrument(1, units = 20000) # close long position and go short 10,000 units

In [ ]:
bc.units

In [ ]:
bc.print_current_balance(1) # cash balance increased, but... 

In [ ]:
bc.print_current_position_value(1) # ... the current position value is negative

In [ ]:
bc.print_current_nav(1)

## Creating an Iterative Base Class (Part 7)

In [39]:
class IterativeBase():

    def __init__(self, symbol, start, end, amount):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.initial_balance = amount
        self.current_balance = amount
        self.units = 0
        self.trades = 0 
        self.get_data()

    def get_data(self):
        raw = pd.read_csv("detailed.csv", parse_dates = ["time"], index_col = "time").dropna()
        raw = raw.loc[self.start:self.end]
        raw["returns"] = np.log(raw.price / raw.price.shift(1))
        self.data = raw

    def plot_data(self, cols = None):  
        if cols is None:
            cols = "price"
        self.data[cols].plot(figsize = (12, 8), title = self.symbol)
    
    def get_values(self, bar):
        date = str(self.data.index[bar].date())
        price = round(self.data.price.iloc[bar], 5)
        spread = round(self.data.spread.iloc[bar], 5)
        return date, price, spread
    
    def print_current_balance(self, bar):
        date, price, spread = self.get_values(bar)
        print("{} | Current Balance: {}".format(date, round(self.current_balance, 2)))
        
    def buy_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance -= units * price # reduce cash balance by "purchase price"
        self.units += units
        self.trades += 1
        print("{} |  Buying {} for {}".format(date, units, round(price, 5)))
    
    def sell_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance += units * price # increases cash balance by "purchase price"
        self.units -= units
        self.trades += 1
        print("{} |  Selling {} for {}".format(date, units, round(price, 5)))
    
    def print_current_position_value(self, bar):
        date, price, spread = self.get_values(bar)
        cpv = self.units * price
        print("{} |  Current Position Value = {}".format(date, round(cpv, 2)))
    
    def print_current_nav(self, bar):
        date, price, spread = self.get_values(bar)
        nav = self.current_balance + self.units * price
        print("{} |  Net Asset Value = {}".format(date, round(nav, 2)))
        
    def close_pos(self, bar):
        date, price, spread = self.get_values(bar)
        print(75 * "-")
        print("{} | +++ CLOSING FINAL POSITION +++".format(date))
        self.current_balance += self.units * price # closing final position (works with short and long!)
        print("{} | closing position of {} for {}".format(date, self.units, price))
        self.units = 0 # setting position to neutral
        self.trades += 1
        perf = (self.current_balance - self.initial_balance) / self.initial_balance * 100
        self.print_current_balance(bar)
        print("{} | net performance (%) = {}".format(date, round(perf, 2) ))
        print("{} | number of trades executed = {}".format(date, self.trades))
        print(75 * "-")

In [40]:
bc = IterativeBase("EURUSD", "2006-12-31", "2020-06-30", 100000)

In [41]:
bc.buy_instrument(0, amount = 100000)

2006-12-31 |  Buying 75766 for 1.31985


In [42]:
bc.print_current_balance(0)

2006-12-31 | Current Balance: 0.24


In [43]:
bc.print_current_position_value(0)

2006-12-31 |  Current Position Value = 99999.76


In [44]:
bc.print_current_balance(-1)

2020-06-29 | Current Balance: 0.24


In [45]:
bc.print_current_position_value(-1)

2020-06-29 |  Current Position Value = 85108.71


In [46]:
bc.print_current_nav(-1)

2020-06-29 |  Net Asset Value = 85108.95


In [47]:
bc.close_pos(-1)

---------------------------------------------------------------------------
2020-06-29 | +++ CLOSING FINAL POSITION +++
2020-06-29 | closing position of 75766 for 1.12331
2020-06-29 | Current Balance: 85108.95
2020-06-29 | net performance (%) = -14.89
2020-06-29 | number of trades executed = 2
---------------------------------------------------------------------------


In [48]:
bc.data

,price,spread,returns
time,,,
2006-12-31 22:00:00+00:00,1.31985,0.00100,NaN
2007-01-01 22:00:00+00:00,1.32734,0.00015,0.005659
2007-01-02 22:00:00+00:00,1.31688,0.00015,-0.007912
2007-01-03 22:00:00+00:00,1.30845,0.00015,-0.006422
2007-01-04 22:00:00+00:00,1.30025,0.00100,-0.006287
...,...,...,...
2020-06-23 21:00:00+00:00,1.12507,0.00030,-0.005151
2020-06-24 21:00:00+00:00,1.12180,0.00023,-0.002911
2020-06-25 21:00:00+00:00,1.12184,0.00041,0.000036


In [49]:
bc.data.price.iloc[-1] / bc.data.price.iloc[0] - 1

np.float64(-0.1489108610826988)

## Creating an Iterative Base Class (Part 8)

In [51]:
class IterativeBase():

    def __init__(self, symbol, start, end, amount, use_spread = True):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.initial_balance = amount
        self.current_balance = amount
        self.units = 0
        self.trades = 0
        self.use_spread = use_spread
        self.get_data()

    def get_data(self):
        raw = pd.read_csv("detailed.csv", parse_dates = ["time"], index_col = "time").dropna()
        raw = raw.loc[self.start:self.end]
        raw["returns"] = np.log(raw.price / raw.price.shift(1))
        self.data = raw

    def plot_data(self, cols = None):  
        if cols is None:
            cols = "price"
        self.data[cols].plot(figsize = (12, 8), title = self.symbol)
    
    def get_values(self, bar):
        date = str(self.data.index[bar].date())
        price = round(self.data.price.iloc[bar], 5)
        spread = round(self.data.spread.iloc[bar], 5)
        return date, price, spread
    
    def print_current_balance(self, bar):
        date, price, spread = self.get_values(bar)
        print("{} | Current Balance: {}".format(date, round(self.current_balance, 2)))
        
    def buy_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if self.use_spread:
            price += spread/2 # ask price
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance -= units * price # reduce cash balance by "purchase price"
        self.units += units
        self.trades += 1
        print("{} |  Buying {} for {}".format(date, units, round(price, 5)))
    
    def sell_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if self.use_spread:
            price -= spread/2 # bid price
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance += units * price # increases cash balance by "purchase price"
        self.units -= units
        self.trades += 1
        print("{} |  Selling {} for {}".format(date, units, round(price, 5)))
    
    def print_current_position_value(self, bar):
        date, price, spread = self.get_values(bar)
        cpv = self.units * price
        print("{} |  Current Position Value = {}".format(date, round(cpv, 2)))
    
    def print_current_nav(self, bar):
        date, price, spread = self.get_values(bar)
        nav = self.current_balance + self.units * price
        print("{} |  Net Asset Value = {}".format(date, round(nav, 2)))
        
    def close_pos(self, bar):
        date, price, spread = self.get_values(bar)
        print(75 * "-")
        print("{} | +++ CLOSING FINAL POSITION +++".format(date))
        self.current_balance += self.units * price # closing final position (works with short and long!)
        self.current_balance -= (abs(self.units) * spread/2 * self.use_spread) # substract half-spread costs
        print("{} | closing position of {} for {}".format(date, self.units, price))
        self.units = 0 # setting position to neutral
        self.trades += 1
        perf = (self.current_balance - self.initial_balance) / self.initial_balance * 100
        self.print_current_balance(bar)
        print("{} | net performance (%) = {}".format(date, round(perf, 2) ))
        print("{} | number of trades executed = {}".format(date, self.trades))
        print(75 * "-")

In [52]:
bc = IterativeBase("EURUSD", "2006-12-31", "2020-06-30", 100000, use_spread = True)

In [53]:
bc.buy_instrument(0, amount = 100000)

2006-12-31 |  Buying 75737 for 1.32035


In [54]:
bc.print_current_nav(0)

2006-12-31 |  Net Asset Value = 99962.13


In [55]:
bc.data

,price,spread,returns
time,,,
2006-12-31 22:00:00+00:00,1.31985,0.00100,NaN
2007-01-01 22:00:00+00:00,1.32734,0.00015,0.005659
2007-01-02 22:00:00+00:00,1.31688,0.00015,-0.007912
2007-01-03 22:00:00+00:00,1.30845,0.00015,-0.006422
2007-01-04 22:00:00+00:00,1.30025,0.00100,-0.006287
...,...,...,...
2020-06-23 21:00:00+00:00,1.12507,0.00030,-0.005151
2020-06-24 21:00:00+00:00,1.12180,0.00023,-0.002911
2020-06-25 21:00:00+00:00,1.12184,0.00041,0.000036


In [56]:
bc.data.spread.iloc[0] / 2 * 75737

np.float64(37.8685)

In [57]:
bc.close_pos(-1)

---------------------------------------------------------------------------
2020-06-29 | +++ CLOSING FINAL POSITION +++
2020-06-29 | closing position of 75737 for 1.12331
2020-06-29 | Current Balance: 85064.66
2020-06-29 | net performance (%) = -14.94
2020-06-29 | number of trades executed = 2
---------------------------------------------------------------------------


## Iterative Backtesting of SMA Strategies

In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")

In [59]:
class IterativeBase():

    def __init__(self, symbol, start, end, amount, use_spread = True):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.initial_balance = amount
        self.current_balance = amount
        self.units = 0
        self.trades = 0
        self.position = 0
        self.use_spread = use_spread
        self.get_data()

    def get_data(self):
        raw = pd.read_csv("detailed.csv", parse_dates = ["time"], index_col = "time").dropna()
        raw = raw.loc[self.start:self.end]
        raw["returns"] = np.log(raw.price / raw.price.shift(1))
        self.data = raw

    def plot_data(self, cols = None):  
        if cols is None:
            cols = "price"
        self.data[cols].plot(figsize = (12, 8), title = self.symbol)
    
    def get_values(self, bar):
        date = str(self.data.index[bar].date())
        price = round(self.data.price.iloc[bar], 5)
        spread = round(self.data.spread.iloc[bar], 5)
        return date, price, spread
    
    def print_current_balance(self, bar):
        date, price, spread = self.get_values(bar)
        print("{} | Current Balance: {}".format(date, round(self.current_balance, 2)))
        
    def buy_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if self.use_spread:
            price += spread/2 # ask price
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance -= units * price # reduce cash balance by "purchase price"
        self.units += units
        self.trades += 1
        print("{} |  Buying {} for {}".format(date, units, round(price, 5)))
    
    def sell_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if self.use_spread:
            price -= spread/2 # bid price
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance += units * price # increases cash balance by "purchase price"
        self.units -= units
        self.trades += 1
        print("{} |  Selling {} for {}".format(date, units, round(price, 5)))
    
    def print_current_position_value(self, bar):
        date, price, spread = self.get_values(bar)
        cpv = self.units * price
        print("{} |  Current Position Value = {}".format(date, round(cpv, 2)))
    
    def print_current_nav(self, bar):
        date, price, spread = self.get_values(bar)
        nav = self.current_balance + self.units * price
        print("{} |  Net Asset Value = {}".format(date, round(nav, 2)))
        
    def close_pos(self, bar):
        date, price, spread = self.get_values(bar)
        print(75 * "-")
        print("{} | +++ CLOSING FINAL POSITION +++".format(date))
        self.current_balance += self.units * price # closing final position (works with short and long!)
        self.current_balance -= (abs(self.units) * spread/2 * self.use_spread) # substract half-spread costs
        print("{} | closing position of {} for {}".format(date, self.units, price))
        self.units = 0 # setting position to neutral
        self.trades += 1
        perf = (self.current_balance - self.initial_balance) / self.initial_balance * 100
        self.print_current_balance(bar)
        print("{} | net performance (%) = {}".format(date, round(perf, 2) ))
        print("{} | number of trades executed = {}".format(date, self.trades))
        print(75 * "-")

In [60]:
class IterativeBacktest(IterativeBase):

    # helper method
    def go_long(self, bar, units = None, amount = None):
        if self.position == -1:
            self.buy_instrument(bar, units = -self.units) # if short position, go neutral first
        if units:
            self.buy_instrument(bar, units = units)
        elif amount:
            if amount == "all":
                amount = self.current_balance
            self.buy_instrument(bar, amount = amount) # go long

    # helper method
    def go_short(self, bar, units = None, amount = None):
        if self.position == 1:
            self.sell_instrument(bar, units = self.units) # if long position, go neutral first
        if units:
            self.sell_instrument(bar, units = units)
        elif amount:
            if amount == "all":
                amount = self.current_balance
            self.sell_instrument(bar, amount = amount) # go short

    def test_sma_strategy(self, SMA_S, SMA_L):
        
        # nice printout
        stm = "Testing SMA strategy | {} | SMA_S = {} & SMA_L = {}".format(self.symbol, SMA_S, SMA_L)
        print("-" * 75)
        print(stm)
        print("-" * 75)
        
        # reset 
        self.position = 0  # initial neutral position
        self.trades = 0  # no trades yet
        self.current_balance = self.initial_balance  # reset initial capital
        self.get_data() # reset dataset
        
        # prepare data
        self.data["SMA_S"] = self.data["price"].rolling(SMA_S).mean()
        self.data["SMA_L"] = self.data["price"].rolling(SMA_L).mean()
        self.data.dropna(inplace = True)

        # sma crossover strategy
        for bar in range(len(self.data)-1): # all bars (except the last bar)
            if self.data["SMA_S"].iloc[bar] > self.data["SMA_L"].iloc[bar]: # signal to go long
                if self.position in [0, -1]:
                    self.go_long(bar, amount = "all") # go long with full amount
                    self.position = 1  # long position
            elif self.data["SMA_S"].iloc[bar] < self.data["SMA_L"].iloc[bar]: # signal to go short
                if self.position in [0, 1]:
                    self.go_short(bar, amount = "all") # go short with full amount
                    self.position = -1 # short position
        self.close_pos(bar+1) # close position at the last bar

In [61]:
bc = IterativeBacktest("EURUSD", "2006-12-31", "2020-06-30", 100000, use_spread= True)

In [62]:
bc.data

,price,spread,returns
time,,,
2006-12-31 22:00:00+00:00,1.31985,0.00100,NaN
2007-01-01 22:00:00+00:00,1.32734,0.00015,0.005659
2007-01-02 22:00:00+00:00,1.31688,0.00015,-0.007912
2007-01-03 22:00:00+00:00,1.30845,0.00015,-0.006422
2007-01-04 22:00:00+00:00,1.30025,0.00100,-0.006287
...,...,...,...
2020-06-23 21:00:00+00:00,1.12507,0.00030,-0.005151
2020-06-24 21:00:00+00:00,1.12180,0.00023,-0.002911
2020-06-25 21:00:00+00:00,1.12184,0.00041,0.000036


In [63]:
bc.test_sma_strategy(50, 200)

---------------------------------------------------------------------------
Testing SMA strategy | EURUSD | SMA_S = 50 & SMA_L = 200
---------------------------------------------------------------------------
2007-08-10 |  Buying 72998 for 1.36989
2008-08-28 |  Selling 72998 for 1.46685
2008-08-28 |  Selling 72998 for 1.46685
2009-04-28 |  Buying 72998 for 1.3276
2009-04-28 |  Buying 88311 for 1.3276
2010-01-20 |  Selling 88311 for 1.40833
2010-01-20 |  Selling 88311 for 1.40833
2010-09-27 |  Buying 88311 for 1.35862
2010-09-27 |  Buying 94774 for 1.35862
2011-01-12 |  Selling 94774 for 1.33626
2011-01-12 |  Selling 94774 for 1.33626
2011-01-30 |  Buying 94774 for 1.36946
2011-01-30 |  Buying 90178 for 1.36946
2011-09-10 |  Selling 90178 for 1.35825
2011-09-10 |  Selling 90178 for 1.35825
2012-10-08 |  Buying 90178 for 1.28857
2012-10-08 |  Buying 99931 for 1.28857
2013-04-15 |  Selling 99931 for 1.31755
2013-04-15 |  Selling 99931 for 1.31755
2013-08-04 |  Buying 99931 for 1.32592
201

## Using Modules and adding Docstrings

In [64]:
import IterativeBacktest as IB

In [66]:
bc = IB.IterativeBacktest("EURUSD", "2006-12-31", "2020-06-30", 100000, use_spread = True)

In [67]:
bc.data

,price,spread,returns
time,,,
2006-12-31 22:00:00+00:00,1.31985,0.00100,NaN
2007-01-01 22:00:00+00:00,1.32734,0.00015,0.005659
2007-01-02 22:00:00+00:00,1.31688,0.00015,-0.007912
2007-01-03 22:00:00+00:00,1.30845,0.00015,-0.006422
2007-01-04 22:00:00+00:00,1.30025,0.00100,-0.006287
...,...,...,...
2020-06-23 21:00:00+00:00,1.12507,0.00030,-0.005151
2020-06-24 21:00:00+00:00,1.12180,0.00023,-0.002911
2020-06-25 21:00:00+00:00,1.12184,0.00041,0.000036


In [68]:
bc.test_sma_strategy(50, 200)

---------------------------------------------------------------------------
Testing SMA strategy | EURUSD | SMA_S = 50 & SMA_L = 200
---------------------------------------------------------------------------
2007-08-10 |  Buying 72998 for 1.36989
2008-08-28 |  Selling 72998 for 1.46685
2008-08-28 |  Selling 72998 for 1.46685
2009-04-28 |  Buying 72998 for 1.3276
2009-04-28 |  Buying 88311 for 1.3276
2010-01-20 |  Selling 88311 for 1.40833
2010-01-20 |  Selling 88311 for 1.40833
2010-09-27 |  Buying 88311 for 1.35862
2010-09-27 |  Buying 94774 for 1.35862
2011-01-12 |  Selling 94774 for 1.33626
2011-01-12 |  Selling 94774 for 1.33626
2011-01-30 |  Buying 94774 for 1.36946
2011-01-30 |  Buying 90178 for 1.36946
2011-09-10 |  Selling 90178 for 1.35825
2011-09-10 |  Selling 90178 for 1.35825
2012-10-08 |  Buying 90178 for 1.28857
2012-10-08 |  Buying 99931 for 1.28857
2013-04-15 |  Selling 99931 for 1.31755
2013-04-15 |  Selling 99931 for 1.31755
2013-08-04 |  Buying 99931 for 1.32592
201

## Adding Contrarian and Bollinger Strategy to the Framework

In [70]:
import IterativeBacktest as IB

In [71]:
bc = IB.IterativeBacktest("EURUSD", "2006-12-31", "2020-06-30", 100000, use_spread= True)

In [72]:
bc.data

,price,spread,returns
time,,,
2006-12-31 22:00:00+00:00,1.31985,0.00100,NaN
2007-01-01 22:00:00+00:00,1.32734,0.00015,0.005659
2007-01-02 22:00:00+00:00,1.31688,0.00015,-0.007912
2007-01-03 22:00:00+00:00,1.30845,0.00015,-0.006422
2007-01-04 22:00:00+00:00,1.30025,0.00100,-0.006287
...,...,...,...
2020-06-23 21:00:00+00:00,1.12507,0.00030,-0.005151
2020-06-24 21:00:00+00:00,1.12180,0.00023,-0.002911
2020-06-25 21:00:00+00:00,1.12184,0.00041,0.000036


__Contrarian__

In [73]:
bc.test_con_strategy(window = 3)

---------------------------------------------------------------------------
Testing Contrarian strategy | EURUSD | Window = 3
---------------------------------------------------------------------------
2007-01-03 |  Buying 76421 for 1.30852
2007-01-14 |  Selling 76421 for 1.29364
2007-01-14 |  Selling 76421 for 1.29364
2007-01-15 |  Buying 76421 for 1.29185
2007-01-15 |  Buying 76634 for 1.29185
2007-01-16 |  Selling 76634 for 1.29372
2007-01-16 |  Selling 76634 for 1.29372
2007-01-21 |  Buying 76634 for 1.29507
2007-01-21 |  Buying 76476 for 1.29507
2007-01-22 |  Selling 76476 for 1.30258
2007-01-22 |  Selling 76476 for 1.30258
2007-01-24 |  Buying 76476 for 1.29328
2007-01-24 |  Buying 77575 for 1.29328
2007-01-28 |  Selling 77575 for 1.29561
2007-01-28 |  Selling 77575 for 1.29561
2007-02-01 |  Buying 77575 for 1.29666
2007-02-01 |  Buying 77449 for 1.29666
2007-02-05 |  Selling 77449 for 1.29841
2007-02-05 |  Selling 77449 for 1.29841
2007-02-09 |  Buying 77449 for 1.30135
2007-02-

__Bollinger__

In [74]:
bc.test_boll_strategy(50, 2)

---------------------------------------------------------------------------
Testing Bollinger Bands Strategy | EURUSD | SMA = 50 & dev = 2
---------------------------------------------------------------------------
2007-02-26 |  Selling 75519 for 1.32416
2007-05-16 |  Buying 75519 for 1.34947
2007-06-07 |  Buying 73318 for 1.33784
2007-06-28 |  Selling 73318 for 1.3536
2007-07-01 |  Selling 72850 for 1.36229
2007-08-12 |  Buying 72850 for 1.36137
2007-09-17 |  Selling 71022 for 1.3983
2007-12-13 |  Buying 71022 for 1.44347
2008-02-25 |  Selling 64180 for 1.49738
2008-04-25 |  Buying 64180 for 1.5632
2008-07-01 |  Selling 57852 for 1.58817
2008-07-23 |  Buying 57852 for 1.56784
2008-08-06 |  Buying 60717 for 1.5326
2008-09-21 |  Selling 60717 for 1.47754
2008-10-01 |  Buying 64905 for 1.38221
2008-11-24 |  Selling 64905 for 1.30635
2008-12-10 |  Selling 63508 for 1.33508
2009-01-11 |  Buying 63508 for 1.33632
2009-03-17 |  Selling 62877 for 1.3472
2009-04-16 |  Buying 62877 for 1.30497
